### LangGraph 커스텀 에이전트 설계

---

#### 1. 이름

**부모는 처음이라** (`first-time-parent`)

#### 2. 목적

초보 부모는 아이의 사소한 증상 하나에도 "병원에 가야 하나?" 를 매번 불안해한다. 하지만 인터넷 검색은 과장된 최악의 시나리오로, 지인 조언은 근거 없이 안심시키는 쪽으로 치우치기 쉽다.

이 에이전트는 부모가 아이 상태를 입력하면 **섣불리 진단하지 않되**, 근거 기반 rule로 위험 상황은 놓치지 않고 **보수적으로** 안내한다. 핵심은 두 방향의 실패를 모두 피하는 것이다 — 모든 걸 "위험"으로 몰아 신뢰를 잃는 **과잉 경고("양치기소년")**, 그리고 진짜 응급을 놓치는 **위험 누락**.

#### 3. 핵심 기능

1. **상황 분석 및 구조화** — 부모의 자연어 입력에서 월령·체온·증상 등 판단에 필요한 요소를 추출·정규화한다.
2. **이중 위험도 판단 (LLM + rule guardrail)** — LLM이 정직하게 판단하되, 근거가 확정된 결정론적 rule이 안전측 최소 위험도를 보장한다. 둘 중 **더 높은 위험도를 채택** (`final_risk = max(llm_risk, rule_risk)`) 하여 보수적으로 안내한다.
3. **근거 검색** — 판단의 근거를 공신력 있는 출처(아이사랑·질병관리청·AAP·CDC 등)에서 찾아 함께 제시한다.
4. **의사결정 코치** — "지금 응급실 / 진료 예약 / 집에서 관찰" 처럼 부모가 다음에 무엇을 해야 할지 구체적 행동을 안내한다.

#### 4. 그래프 구조
![](docs/agent_flow.png)

In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool

class State(TypedDict):
    input: str
    month: int
    temp: int
    symptoms: str
    topic: str
    llm_risk: Literal["high", "warn", "low"]    # 신규: LLM 판단 결과
    rule_risk: Literal["high", "warn", "low"]   # 신규: rule 판단 결과
    risk: Literal["high", "warn", "low"]        # aggregate가 채우는 최종값
    source: str
    answer: str

In [ ]:
GUIDE_TABLE = {
    "발열": "아이사랑 - 영유아 발열 대응 가이드 (childcare.go.kr)",
    "수면": "아이사랑 - 안전 수면 원칙 (childcare.go.kr)",
    "수유": "아이사랑 - 월령별 수유량 가이드 (childcare.go.kr)",
}

@tool
def search_official_guide(month: int, topic: str) -> str:
    """월령과 주제로 공식 육아 기관의 근거 페이지를 조회한다."""
    return GUIDE_TABLE.get(topic, "확인된 공식 자료를 찾지 못했습니다.")

In [ ]:
def risk_node(state: State):
    if(state["month"] <= 3 and state["temp"] >= 38):
        return { "risk": "high" }
    else: return { "risk": "low" }

def analyze_node(state: State):
    return {"month": 2, "temp": 39, "symptoms": "발열", "topic": "발열"}

def decide_node(state: State):
    if state["risk"] == "high":
        return {"answer": "🚨 지금 바로 소아청소년과 또는 응급 상담에 연락하세요."}
    else:
        return {"answer": f"📋 {state['symptoms']} 관련 교육 답변입니다. (근거: {state.get('source', '없음')})"}

def search_node(state: State):
    month = state["month"]
    topic = state["topic"]

    result = search_official_guide.invoke({"month": month, "topic": topic})

    return {"source": result}

In [ ]:
def route_by_risk(state: State):
    if state["risk"] == "high":
        return "decide"
    else:
        return "search"

In [ ]:
builder = StateGraph(State)

builder.add_node("risk", risk_node)
builder.add_node("analyze", analyze_node)
builder.add_node("search", search_node)
builder.add_node("decide", decide_node)

builder.add_edge(START, "analyze")
builder.add_edge("analyze", "risk")
builder.add_conditional_edges("risk", route_by_risk, {"decide": "decide", "search": "search"})
builder.add_edge("search", "decide")
builder.add_edge("decide", END)

graph = builder.compile()
result = graph.invoke({"input": "우리 애 2개월인데, 열이 39도에요. 어떻게 하면 좋을까요?"})
print(result)
print(graph.get_graph().draw_ascii())